# 🌱 Green AI Hackathon — Baseline Notebook

**Goal:** Fine-tune a small pretrained language model to classify movie
review sentiment, measure its energy consumption, then **explore** ways to
reduce that energy, trying as many ideas as you have time for, and
measure each one.

**Task:** This uses **fine-tuning**, which is the dominant real-world
pattern in modern NLP/Green AI: instead of training a model from scratch,
you start from a model that has already learned general language
understanding (DistilBERT, a compact version of BERT), and adjust it
slightly for a specific task; here, deciding whether a movie review is
positive or negative. Fine-tuning a pretrained model is *much* cheaper than
training one from scratch, but at the scale used by real companies, it
still consumes meaningful energy; which is exactly why techniques to
reduce that energy matter in practice, not just in theory.

**Dataset:** the **IMDb Large Movie Review Dataset** (Maas et al., 2011), a
long-standing benchmark of 50,000 movie reviews labeled positive/negative;
also the dataset DistilBERT's creators themselves used to evaluate it
against full BERT.

**You do not need to know how transformer architectures work internally**
to do this hackathon. Every technique here works through a config value,
a checkpoint name, or a function call, so, no model architecture code
required.

**Before you touch anything:**
1. Go to `Runtime > Change runtime type` and select **GPU (T4)** if
   available. It still runs on CPU, just slower (see Section 1).
2. Run the cells in order, top to bottom.
3. Do **not** change Section 1 (Setup) or Section 2 (Baseline). These must
   stay identical across all groups so results are comparable.
4. **Section 3 is your exploration space.** You'll be given a *starting
   direction*, but within it you're free to try multiple ideas, combine
   them, and iterate. Every attempt gets logged automatically.
5. At the end, copy your best numbers into the shared results sheet (link
   shared during workshop).

---


## Section 0 — Install dependencies (run once)

This installs `codecarbon` (for energy tracking), `transformers` and
`datasets` (for the pretrained model and the IMDb dataset), and verifies
`torch` is available. Colab and Kaggle normally ship `torch` preinstalled;
`transformers`/`datasets` usually need installing.

In [ ]:
# --- codecarbon (energy tracking) + transformers/datasets (model + data) ---
!pip install codecarbon transformers datasets -q

# --- torch: install only if missing ---
import importlib.util

if importlib.util.find_spec("torch") is None:
    print("torch not found — installing (this may take a minute)...")
    !pip install torch --index-url https://download.pytorch.org/whl/cpu -q
else:
    print("torch already available.")

# --- Block torchvision from being imported (fixes a VideoReader crash) ---
# This notebook does pure text classification and never needs torchvision
# at all. However, if it gets imported (transformers pulls it in
# opportunistically as a side effect), the `datasets` library's internal
# tensor-conversion code checks `"torchvision" in sys.modules` and then
# tries `from torchvision.io import VideoReader` — a class that newer
# torchvision versions removed entirely. That crashes mid-training with
# "ImportError: cannot import name 'VideoReader' from 'torchvision.io'",
# even though nothing here ever touches video.
#
# The fix has to actually keep "torchvision" OUT of sys.modules (setting
# it to None still leaves the key present, which doesn't help — the check
# above is `in sys.modules`, not whether the value is usable). A custom
# import hook intercepts any attempt to import torchvision and raises
# ImportError immediately, so it's never added to sys.modules at all.
import sys

# Remove torchvision if anything already imported it earlier in this
# session (e.g. a previous cell run), then block it from being imported
# again — covers both "already loaded" and "not yet loaded" cases.
for _mod_name in list(sys.modules.keys()):
    if _mod_name == "torchvision" or _mod_name.startswith("torchvision."):
        del sys.modules[_mod_name]

class _BlockTorchvision:
    def find_module(self, name, path=None):
        if name == "torchvision" or name.startswith("torchvision."):
            return self
        return None

    def load_module(self, name):
        raise ImportError(f"{name} is intentionally blocked by this notebook "
                           f"(not needed for text-only models, and avoids a "
                           f"known torchvision.io.VideoReader crash)")

if not any(isinstance(finder, _BlockTorchvision) for finder in sys.meta_path):
    sys.meta_path.insert(0, _BlockTorchvision())

import torch
import codecarbon
import transformers

# Quiet down transformers' weight-loading report (the "LOAD REPORT" block
# listing UNEXPECTED/MISSING keys every time a model is loaded). This is
# normal, expected output whenever you attach a classification head to a
# pretrained checkpoint — not a sign of a problem — but it gets repetitive
# since load_model() is called many times throughout this notebook.
transformers.logging.set_verbosity_error()

print("Done. torch:", torch.__version__, "| transformers:", transformers.__version__)
print("GPU visible to torch:", torch.cuda.is_available())

torch already available.
Done. torch: 2.11.0+cu128 | transformers: 5.12.1
GPU visible to torch: True


## Section 1 — Setup (DO NOT MODIFY)

This loads a small subset of the IMDb dataset, tokenizes it for DistilBERT,
and defines the training/evaluation functions. Keeping this identical
across groups is what makes your results comparable at the end.

The config below **auto-adjusts to your hardware**: GPU groups get a
larger subset; CPU groups get a smaller subset so things still finish in a
few minutes. This means CPU-tier and GPU-tier groups won't have identical
absolute numbers; what matters is the **relative change** your own
exploration achieves versus your own baseline.

**Every measurement is repeated `N_REPEATS` times and averaged.** A single
energy reading has real run-to-run noise (background load, sampling
granularity, etc.); repeating each run and looking at the mean (and
spread) is what lets you tell a genuine effect apart from measurement
noise. This is also why the dataset subset is smaller than you might
expect: repeating 3 times costs 3x the time, so the per-run size was
reduced to keep the overall time budget similar.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
import numpy as np
import time
import random

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

BASE_MODEL_NAME = "distilbert-base-uncased"
MAX_LENGTH = 128  # caps review length — one of the "shorter input" ideas in Section 3

# N_REPEATS: every measurement (baseline and every attempt) is repeated this
# many times and averaged, because a single codecarbon reading has real
# run-to-run variance (background load, sampling granularity, etc.) — with
# only one run, you can't tell a genuine effect apart from noise.
N_REPEATS = 3

if device.type == "cuda":
    TRAIN_SUBSET_SIZE, TEST_SUBSET_SIZE, BASE_EPOCHS = 500, 250, 2
    print(f"GPU detected: using full config (500 train / 250 test examples, 2 epochs, x{N_REPEATS} repeats).")
else:
    TRAIN_SUBSET_SIZE, TEST_SUBSET_SIZE, BASE_EPOCHS = 150, 75, 2
    print(f"No GPU detected: using reduced config (150 train / 75 test examples, 2 epochs, x{N_REPEATS} repeats)")
    print("so this still finishes in a reasonable time on CPU.")


Using device: cuda
GPU detected: using full config (500 train / 250 test examples, 2 epochs, x3 repeats).


In [ ]:
# --- Load IMDb (Maas et al., 2011) and take a small, fixed-seed subset ---
from datasets import load_dataset

# The IMDb dataset's canonical Hub location is "stanfordnlp/imdb" — newer
# versions of the `datasets` library require this full namespace/name form
# and no longer auto-resolve the older bare "imdb" identifier. We try the
# current path first, with the legacy name as a fallback in case of a
# library version difference.
try:
    imdb = load_dataset("stanfordnlp/imdb")
    print("Loaded dataset as 'stanfordnlp/imdb'.")
except Exception as e:
    print(f"'stanfordnlp/imdb' failed ({e}), trying legacy 'imdb' identifier...")
    imdb = load_dataset("imdb")
    print("Loaded dataset as 'imdb'.")

train_full = imdb["train"].shuffle(seed=SEED)
test_full = imdb["test"].shuffle(seed=SEED)

train_raw = train_full.select(range(TRAIN_SUBSET_SIZE))
test_raw = test_full.select(range(TEST_SUBSET_SIZE))

print(f"Train size: {len(train_raw)} | Test size: {len(test_raw)}")
print(f"Example review (truncated): {train_raw[0]['text'][:200]}...")
print(f"Label: {train_raw[0]['label']} (0=negative, 1=positive)")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loaded dataset as 'stanfordnlp/imdb'.
Train size: 500 | Test size: 250
Example review (truncated): There is no relation at all between Fortier and Profiler but the fact that both are police series about violent crimes. Profiler looks crispy, Fortier looks classic. Profiler plots are quite simple. F...
Label: 1 (0=negative, 1=positive)


In [ ]:
# --- Tokenize for DistilBERT ---
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME)

def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, padding="max_length", max_length=MAX_LENGTH)

train_tokenized = train_raw.map(tokenize_function, batched=True)
test_tokenized = test_raw.map(tokenize_function, batched=True)

train_tokenized = train_tokenized.with_format("torch", columns=["input_ids", "attention_mask", "label"])
test_tokenized = test_tokenized.with_format("torch", columns=["input_ids", "attention_mask", "label"])

train_set = train_tokenized
test_set = test_tokenized

print("Tokenization done.")


Tokenization done.


In [ ]:
# --- Model loader (kept fixed for the baseline) ---
# Note: model_name is a knob — swap this string for a different pretrained
# checkpoint to try a smaller model, no architecture code needed.
from transformers import AutoModelForSequenceClassification, AutoConfig, BertConfig

def load_model(model_name=BASE_MODEL_NAME):
    try:
        return AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)
    except ValueError as e:
        # A few small community checkpoints (e.g. prajjwal1/bert-mini) have a
        # config.json that's missing the "model_type" field, which makes
        # AutoModel unable to auto-detect the architecture. These checkpoints
        # are still standard BERT architectures under the hood, so we can
        # load the config explicitly as BertConfig and retry.
        if "model_type" in str(e):
            print(f"Note: '{model_name}' has an incomplete config.json (missing model_type).")
            print("Retrying by loading it explicitly as a BERT architecture...")
            config = BertConfig.from_pretrained(model_name, num_labels=2)
            return AutoModelForSequenceClassification.from_pretrained(model_name, config=config)
        raise

def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

def safe_mean_std(values):
    """
    Mean and std of a list of readings, skipping any that are None.
    codecarbon can very occasionally fail to produce a reading for a single
    run — this avoids one bad run crashing the whole averaged result.
    """
    clean = [v for v in values if v is not None]
    if len(clean) < len(values):
        print(f"Note: {len(values) - len(clean)} of {len(values)} run(s) had no energy reading "
              f"and were excluded from the average.")
    if not clean:
        return None, None
    return float(np.mean(clean)), float(np.std(clean))


In [ ]:
# --- Fixed training/eval functions used by both baseline and your experiments ---
def train_one_epoch(model, loader, optimizer, device, use_amp=False, scaler=None):
    model.train()
    total_loss = 0.0
    amp_device_type = device.type
    for batch in loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)
        optimizer.zero_grad()
        if use_amp:
            with torch.amp.autocast(amp_device_type):
                outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
                loss = outputs.loss
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            loss = outputs.loss
            loss.backward()
            optimizer.step()
        total_loss += loss.item() * input_ids.size(0)
    return total_loss / len(loader.dataset)

def evaluate(model, loader, device):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["label"].to(device)
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            predicted = outputs.logits.argmax(dim=-1)
            correct += (predicted == labels).sum().item()
            total += labels.size(0)
    return correct / total


## Section 2 — Baseline run (DO NOT MODIFY)

This is the reference point your exploration will be measured against. It
trains and measures the baseline `N_REPEATS` times (3, by default) and
averages the results — run it once, record the numbers it prints (also
saved to `baseline_results.json`), then move to Section 3.

Baseline config: `distilbert-base-uncased`, batch size 16, fp32,
`BASE_EPOCHS` epochs, repeated `N_REPEATS` times.

In [ ]:
from codecarbon import EmissionsTracker
import json

BASELINE_CONFIG = dict(batch_size=16, epochs=BASE_EPOCHS, lr=2e-5)

run_accuracies, run_times, run_energies, run_emissions = [], [], [], []
num_params_baseline = None

for run_idx in range(N_REPEATS):
    train_loader = DataLoader(train_set, batch_size=BASELINE_CONFIG["batch_size"], shuffle=True)
    test_loader = DataLoader(test_set, batch_size=32, shuffle=False)

    model = load_model().to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=BASELINE_CONFIG["lr"])
    num_params_baseline = count_params(model)

    tracker = EmissionsTracker(measure_power_secs=5, save_to_file=False, log_level="error")
    tracker.start()
    start_time = time.time()

    for epoch in range(BASELINE_CONFIG["epochs"]):
        loss = train_one_epoch(model, train_loader, optimizer, device)
    print(f"[baseline run {run_idx+1}/{N_REPEATS}] final loss {loss:.4f}")

    elapsed = time.time() - start_time
    emissions_kg = tracker.stop()
    accuracy = evaluate(model, test_loader, device)
    energy_kwh = tracker._total_energy.kWh if hasattr(tracker, "_total_energy") else None

    run_accuracies.append(accuracy)
    run_times.append(elapsed)
    run_energies.append(energy_kwh)
    run_emissions.append(emissions_kg)

    del model, optimizer
    if device.type == "cuda":
        torch.cuda.empty_cache()

energy_mean, energy_std = safe_mean_std(run_energies)
emissions_mean, emissions_std = safe_mean_std(run_emissions)

baseline_results = {
    "config": BASELINE_CONFIG,
    "model_name": BASE_MODEL_NAME,
    "num_params": num_params_baseline,
    "n_repeats": N_REPEATS,
    "accuracy": float(np.mean(run_accuracies)),
    "accuracy_std": float(np.std(run_accuracies)),
    "train_time_sec": float(np.mean(run_times)),
    "train_time_sec_std": float(np.std(run_times)),
    "energy_kwh": energy_mean,
    "energy_kwh_std": energy_std,
    "emissions_kg_co2": emissions_mean,
    "emissions_kg_co2_std": emissions_std,
    "all_runs": {
        "accuracy": run_accuracies, "train_time_sec": run_times,
        "energy_kwh": run_energies, "emissions_kg_co2": run_emissions,
    },
}

print(f"\n=== BASELINE RESULTS (mean over {N_REPEATS} runs — write these down!) ===")
print(f"Accuracy:        {baseline_results['accuracy']*100:.2f}% (std: {baseline_results['accuracy_std']*100:.2f}%)")
print(f"Train time:      {baseline_results['train_time_sec']:.1f} sec (std: {baseline_results['train_time_sec_std']:.1f})")
if baseline_results['energy_kwh'] is not None:
    print(f"Energy used:     {baseline_results['energy_kwh']:.6f} kWh (std: {baseline_results['energy_kwh_std']:.6f})")
else:
    print("Energy used:     no valid reading from any run — try re-running this cell.")
if baseline_results['emissions_kg_co2'] is not None:
    print(f"CO2 emissions:   {baseline_results['emissions_kg_co2']*1000:.4f} g CO2eq "
          f"(std: {baseline_results['emissions_kg_co2_std']*1000:.4f} g)")
else:
    print("CO2 emissions:   no valid reading from any run.")
print(f"Params:          {baseline_results['num_params']:,}")
print(f"\nIndividual runs (energy kWh): {[f'{e:.6f}' if e is not None else 'None' for e in run_energies]}")
print("If these vary quite a bit from each other, that's the measurement noise this repetition is meant to reveal.")

with open("baseline_results.json", "w") as f:
    json.dump(baseline_results, f, indent=2)


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[baseline run 1/3] final loss 0.4148


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[baseline run 2/3] final loss 0.4115


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[baseline run 3/3] final loss 0.5047

=== BASELINE RESULTS (mean over 3 runs — write these down!) ===
Accuracy:        78.80% (std: 4.83%)
Train time:      11.3 sec (std: 0.6)
Energy used:     0.000262 kWh (std: 0.000009)
CO2 emissions:   0.0915 g CO2eq (std: 0.0030 g)
Params:          66,955,010

Individual runs (energy kWh): ['0.000273', '0.000260', '0.000253']
If these vary quite a bit from each other, that's the measurement noise this repetition is meant to reveal.


## Section 3 — Explore! (THIS IS YOUR SPACE)

This is the open-ended part of the hackathon. There's no assigned task,
explore freely and try whatever you're curious about. You are encouraged
to:

- Try **more than one idea**, not just the first thing that comes to mind
- **Combine** ideas (e.g. a smaller pretrained model *and* fewer epochs)
- Use the **leaderboard cell** later in this section to compare every
  attempt and pick your best trade-off for the presentation

There's no single "right" answer. The interesting part is *characterizing
the trade-off* between energy saved and accuracy lost (or gained).

**You don't need to know how transformers work internally to do this.**
Every idea below works through a single setting, checkpoint name, or
function call, no model architecture code required.

### Ideas to try (pick any, combine them, or come up with your own)

| Idea | What to change | Why it might save energy |
|---|---|---|
| Smaller pretrained model | Swap the checkpoint name: `load_model("prajjwal1/bert-tiny")` or `"prajjwal1/bert-mini"` instead of the DistilBERT default | A smaller pretrained model has fewer parameters and FLOPs per pass |
| Mixed precision | Set `use_amp=True` in `run_experiment()` | Less data movement, faster ops on GPU tensor cores |
| Fewer epochs | Lower `epochs` in your config | Less total compute, at the cost of convergence |
| Batch size | Try `batch_size` 8 / 16 / 32 / 64 | Bigger batches can use hardware more efficiently per sample |
| Shorter input / less data | Lower `MAX_LENGTH` (e.g. to 64) and re-run Section 1, or train on a smaller `TRAIN_SUBSET_SIZE` | Shorter sequences and fewer examples both cut compute per pass |
| Your own idea | Anything you can change with a setting | Explain your hypothesis when you present |

### How to run an experiment

Use the `run_experiment(...)` helper below. It wraps training + evaluation
+ energy tracking in one call and **automatically logs every attempt** to
`my_attempts`.

In [ ]:
# ============================================================
# Reusable experiment runner — you don't need to modify this cell.
#
# Note: model_fn is a FUNCTION that returns a fresh model, not a model
# itself — e.g. model_fn=lambda: load_model("prajjwal1/bert-tiny").to(device)
# This is needed because every attempt is repeated N_REPEATS times with a
# brand new model each time (reusing one already-trained model across
# "repeats" would just keep training it further, not measure independent runs).
# ============================================================

my_attempts = []  # every call to run_experiment() appends a result here

def run_experiment(label, model_fn, config, train_dataset=train_set, test_dataset=test_set,
                    use_amp=False, custom_train_loop=None, n_repeats=None):
    """
    Build a fresh model via model_fn(), train it with `config` (dict with
    batch_size, epochs, lr), track energy with codecarbon, evaluate, and
    repeat n_repeats times (default: N_REPEATS). Logs the averaged result
    into my_attempts.
    """
    n_repeats = n_repeats or N_REPEATS
    run_accuracies, run_times, run_energies, run_emissions = [], [], [], []
    num_params_run = None

    for run_idx in range(n_repeats):
        model = model_fn()
        train_loader_exp = DataLoader(train_dataset, batch_size=config["batch_size"], shuffle=True)
        test_loader_exp = DataLoader(test_dataset, batch_size=32, shuffle=False)

        optimizer = torch.optim.AdamW(model.parameters(), lr=config.get("lr", 2e-5))
        num_params_run = count_params(model)

        use_amp_this_run = use_amp
        if use_amp_this_run and device.type != "cuda":
            if run_idx == 0:
                print("Note: use_amp=True has no effect without a GPU — running in normal fp32 instead.")
            use_amp_this_run = False

        scaler = torch.amp.GradScaler("cuda") if use_amp_this_run else None

        tracker_exp = EmissionsTracker(measure_power_secs=5, save_to_file=False, log_level="error")
        tracker_exp.start()
        start = time.time()

        if custom_train_loop is not None:
            custom_train_loop(model, train_loader_exp, optimizer, device)
        else:
            for epoch in range(config["epochs"]):
                loss = train_one_epoch(model, train_loader_exp, optimizer, device,
                                        use_amp=use_amp_this_run, scaler=scaler)
            print(f"[{label}] run {run_idx+1}/{n_repeats} - final loss {loss:.4f}")

        elapsed = time.time() - start
        emissions_kg = tracker_exp.stop()
        accuracy = evaluate(model, test_loader_exp, device)
        energy_kwh = tracker_exp._total_energy.kWh if hasattr(tracker_exp, "_total_energy") else None

        run_accuracies.append(accuracy)
        run_times.append(elapsed)
        run_energies.append(energy_kwh)
        run_emissions.append(emissions_kg)

        del model, optimizer
        if device.type == "cuda":
            torch.cuda.empty_cache()

    energy_mean, energy_std = safe_mean_std(run_energies)
    emissions_mean, emissions_std = safe_mean_std(run_emissions)

    result = {
        "label": label, "config": config, "num_params": num_params_run,
        "n_repeats": n_repeats,
        "accuracy": float(np.mean(run_accuracies)), "accuracy_std": float(np.std(run_accuracies)),
        "train_time_sec": float(np.mean(run_times)), "train_time_sec_std": float(np.std(run_times)),
        "energy_kwh": energy_mean, "energy_kwh_std": energy_std,
        "emissions_kg_co2": emissions_mean, "emissions_kg_co2_std": emissions_std,
        "all_runs": {
            "accuracy": run_accuracies, "train_time_sec": run_times,
            "energy_kwh": run_energies, "emissions_kg_co2": run_emissions,
        },
    }
    my_attempts.append(result)

    print(f"\n=== Result: {label} (mean over {n_repeats} runs) ===")
    print(f"Accuracy:      {result['accuracy']*100:.2f}% (std: {result['accuracy_std']*100:.2f}%)")
    print(f"Train time:    {result['train_time_sec']:.1f} sec (std: {result['train_time_sec_std']:.1f})")
    if result['energy_kwh'] is not None:
        print(f"Energy used:   {result['energy_kwh']:.6f} kWh (std: {result['energy_kwh_std']:.6f})")
    else:
        print("Energy used:   no valid reading from any run — try this attempt again.")
    if result['emissions_kg_co2'] is not None:
        print(f"CO2 emissions: {result['emissions_kg_co2']*1000:.4f} g CO2eq (std: {result['emissions_kg_co2_std']*1000:.4f} g)")
    else:
        print("CO2 emissions: no valid reading from any run.")
    print(f"Params:        {result['num_params']:,}")

    return result


In [ ]:
# ============================================================
# YOUR TURN — pick an idea from the table above (or your own), set a
# config, optionally change the checkpoint name, call run_experiment(...).
# Call this cell (or copies of it) as many times as you want.
#
# model_fn is a function that returns a fresh model each time it's called
# (needed for repeated runs) — wrap your model-loading line in a lambda.
# ============================================================

my_model_fn = lambda: load_model(BASE_MODEL_NAME).to(device)  # <- change the checkpoint name to try a different pretrained model
my_config = dict(batch_size=16, epochs=BASE_EPOCHS, lr=2e-5)  # <- change batch_size / epochs / lr

run_experiment(
    label="CHANGE_ME: describe what you tried",
    model_fn=my_model_fn,
    config=my_config,
    use_amp=False,   # <- set True to try mixed precision
)


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[CHANGE_ME: describe what you tried] run 1/3 - final loss 0.4819


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[CHANGE_ME: describe what you tried] run 2/3 - final loss 0.4490


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[CHANGE_ME: describe what you tried] run 3/3 - final loss 0.4424

=== Result: CHANGE_ME: describe what you tried (mean over 3 runs) ===
Accuracy:      80.27% (std: 1.64%)
Train time:    10.9 sec (std: 0.4)
Energy used:   0.000254 kWh (std: 0.000012)
CO2 emissions: 0.0887 g CO2eq (std: 0.0043 g)
Params:        66,955,010


{'label': 'CHANGE_ME: describe what you tried',
 'config': {'batch_size': 16, 'epochs': 2, 'lr': 2e-05},
 'num_params': 66955010,
 'n_repeats': 3,
 'accuracy': 0.8026666666666668,
 'accuracy_std': 0.01643843734125057,
 'train_time_sec': 10.879194657007853,
 'train_time_sec_std': 0.35768944145418946,
 'energy_kwh': 0.00025392599638729597,
 'energy_kwh_std': 1.2254862011268638e-05,
 'emissions_kg_co2': 8.867761302684575e-05,
 'emissions_kg_co2_std': 4.279718999212488e-06,
 'all_runs': {'accuracy': [0.8, 0.824, 0.784],
  'train_time_sec': [10.583532810211182,
   10.67156720161438,
   11.382483959197998],
  'energy_kwh': [0.00024335112884806446,
   0.00024732218981922475,
   0.0002711046704945987],
  'emissions_kg_co2': [8.498459212785975e-05,
   8.63713906956183e-05,
   9.467685625705917e-05]}}

In [ ]:
# ============================================================
# Try a SECOND idea (or a combination). Add more cells like this freely —
# every call appends to my_attempts, nothing is overwritten.
# ============================================================

# my_model_fn_2 = lambda: load_model(...).to(device)
# my_config_2 = dict(batch_size=..., epochs=..., lr=...)
# run_experiment(label="...", model_fn=my_model_fn_2, config=my_config_2, use_amp=...)


### Reuse, don't recompute (talking point)
Worth a sentence in your presentation if relevant: in real pipelines,
caching tokenized data (rather than re-tokenizing every run) and resuming
from checkpoints instead of restarting failed fine-tuning runs from
scratch avoids paying the same energy cost twice. This notebook does a
version of this too, the baseline is computed once and reused as the
comparison point for every attempt.

## Section 4 — Final comparison & report

Pick your **best attempt** (best trade-off, not necessarily lowest energy
if accuracy dropped too much). Run this cell to
get the exact numbers to copy into the shared results sheet.

### Leaderboard — see all your attempts side by side

Run this any time to compare everything you've tried so far against the
baseline — accuracy, energy (with its standard deviation across the
repeated runs), and training time. The std column is worth looking at: if
it's large relative to the difference between two attempts, that
difference might just be measurement noise, not a real effect. Use this to
decide which attempt is your best trade-off to present.

In [ ]:
import json

with open("baseline_results.json") as f:
    baseline_results = json.load(f)

def pct_change(new, old):
    if old in (0, None) or new is None:
        return float("nan")
    return (new - old) / old * 100

def fmt(value, spec=".6f", width=13):
    # Safely formats a number, or a placeholder if it's None (can happen
    # in the rare case codecarbon failed to produce a reading for every
    # repeat of one attempt).
    if value is None:
        return f"{'n/a':<{width}}"
    return f"{value:<{width}{spec}}"

print(f"{'Attempt':<45}{'Acc %':<9}{'Acc D%':<9}{'Energy kWh':<13}{'+-std':<11}{'Energy D%':<11}{'Time s':<9}{'Time D%':<9}")
print("-" * 116)
print(f"{'BASELINE':<45}{baseline_results['accuracy']*100:<9.2f}{'--':<9}"
      f"{fmt(baseline_results['energy_kwh'])}{fmt(baseline_results['energy_kwh_std'], width=11)}{'--':<11}"
      f"{baseline_results['train_time_sec']:<9.1f}{'--':<9}")

for att in my_attempts:
    acc_delta = pct_change(att['accuracy'], baseline_results['accuracy'])
    energy_delta = pct_change(att['energy_kwh'], baseline_results['energy_kwh'])
    time_delta = pct_change(att['train_time_sec'], baseline_results['train_time_sec'])
    energy_delta_str = f"{energy_delta:+.1f}" if energy_delta == energy_delta else "n/a"  # NaN check
    print(f"{att['label'][:44]:<45}{att['accuracy']*100:<9.2f}{acc_delta:<+9.1f}"
          f"{fmt(att['energy_kwh'])}{fmt(att['energy_kwh_std'], width=11)}{energy_delta_str:<11}"
          f"{att['train_time_sec']:<9.1f}{time_delta:<+9.1f}")

if my_attempts:
    best = min(my_attempts, key=lambda a: a['energy_kwh'] if a['energy_kwh'] is not None else float('inf'))
    print(f"\nLowest-energy attempt so far: '{best['label']}' "
          f"({pct_change(best['energy_kwh'], baseline_results['energy_kwh']):+.1f}% energy, "
          f"{pct_change(best['accuracy'], baseline_results['accuracy']):+.1f}% accuracy, "
          f"{pct_change(best['train_time_sec'], baseline_results['train_time_sec']):+.1f}% time)")


Attempt                                      Acc %    Acc D%   Energy kWh   +-std      Energy D%  Time s   Time D%  
--------------------------------------------------------------------------------------------------------------------
BASELINE                                     78.80    --       0.000262     0.000009   --         11.3     --       
CHANGE_ME: describe what you tried           80.27    +1.9     0.000254     0.000012   -3.1       10.9     -3.5     

Lowest-energy attempt so far: 'CHANGE_ME: describe what you tried' (-3.1% energy, +1.9% accuracy, -3.5% time)


---
### Reminder for your 2-min presentation
1. What did you try, and why did you expect it to save energy?
2. What happened to accuracy vs. energy across your attempts? Was there a
   clear trade-off, or a technique that came with little or no accuracy
   cost?
3. If you tried more than one idea, which one would you actually ship, and
   why?
4. Would you deploy your best result in production? What would you want to
   test further first?

### About the dataset and model
The IMDb Large Movie Review Dataset (Maas, Daly, Pham, Huang, Ng & Potts,
2011) is a long-standing benchmark of 50,000 polarized movie reviews,
widely used for sentiment classification research. DistilBERT (Sanh,
Debut, Chaumond & Wolf, 2019) is a compact, distilled version of BERT,
40% smaller and 60% faster, while retaining about 97% of BERT's language
understanding.
